## Cross-Country Wage Growth Comparison (2019–2023)

SWE wage growth vs other roles across USA, India, and China. Growth in local currency (not PPP-adjusted) to show real domestic trend.

In [ ]:
import sys
from pathlib import Path
ROOT = Path().resolve().parent
sys.path.insert(0, str(ROOT / 'src'))
import pandas as pd
import plotly.express as px

ROLE_LABELS = {
    'software_engineer': 'Software Engineer', 'lawyer': 'Lawyer',
    'physician': 'Physician', 'financial_analyst': 'Financial Analyst',
    'registered_nurse': 'Registered Nurse', 'civil_engineer': 'Civil Engineer',
    'construction_laborer': 'Construction Laborer', 'farm_worker': 'Farm Worker',
    'manufacturing_worker': 'Manufacturing Worker', 'retail_worker': 'Retail Worker',
}
YEARS = [2019, 2020, 2021, 2022, 2023]
country_colors = {'USA': '#2171b5', 'India': '#31a354', 'China': '#e6550d'}

usa = pd.read_csv(ROOT / 'data' / 'processed' / 'merged_usa_data.csv')
india = pd.read_csv(ROOT / 'data' / 'processed' / 'merged_india_data.csv')
china = pd.read_csv(ROOT / 'data' / 'processed' / 'merged_china_data.csv')
df = pd.concat([usa, india, china], ignore_index=True)
grw = df[df['career_stage'] == 'mid'].copy()
grw['role_label'] = grw['role'].map(ROLE_LABELS)

In [ ]:
swe = grw[grw['role'] == 'software_engineer']
rows = []
for _, row in swe.iterrows():
    base = row['wage_2019']
    for yr in YEARS:
        rows.append({'country': row['country'], 'year': yr,
                     'wage_index': row[f'wage_{yr}'] / base * 100})
swe_long = pd.DataFrame(rows)

fig = px.line(
    swe_long, x='year', y='wage_index', color='country', markers=True,
    color_discrete_map=country_colors,
    title='Software Engineer Wage Growth by Country, 2019–2023 (2019 = 100, local currency)',
    labels={'wage_index': 'Wage Index (2019=100)', 'year': 'Year', 'country': 'Country'},
)
fig.add_hline(y=100, line_dash='dot', line_color='grey')
fig.show()

In [ ]:
heat_rows = []
for _, row in grw.iterrows():
    pct = (row['wage_2023'] - row['wage_2019']) / row['wage_2019'] * 100
    heat_rows.append({'country': row['country'], 'role_label': row['role_label'], 'growth_pct': round(pct, 1)})
heat_df = pd.DataFrame(heat_rows)
heat_pivot = heat_df.pivot(index='role_label', columns='country', values='growth_pct')

fig2 = px.imshow(
    heat_pivot, text_auto=True, aspect='auto',
    color_continuous_scale='RdYlGn',
    title='Wage Growth 2019→2023 by Role and Country (%, local currency)',
    labels={'color': 'Growth (%)'},
)
fig2.show()